In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import os
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

print('TensorFlow version:', tf.__version__)

In [ ]:

# Define paths
train_path = r'D:\AI_Projects\EIS\Data_Sets\train'
test_path = r'D:\AI_Projects\EIS\Data_Sets\test'

# Emotion labels
emotion_labels = {
    0: 'Angry',
    1: 'Disgust',
    2: 'Fear',
    3: 'Happy',
    4: 'Sad',
    5: 'Surprise',
    6: 'Neutral'
}

def load_dataset(dataset_path):
    images = []
    labels = []

    for emotion_num, emotion_name in emotion_labels.items():
        emotion_path = os.path.join(dataset_path, emotion_name)

        # Check if directory exists
        if not os.path.isdir(emotion_path):
            print(f"Warning: Directory {emotion_path} not found. Skipping...")
            continue

        for img_file in os.listdir(emotion_path):
            if img_file.endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(emotion_path, img_file)

                try:
                    # Read and preprocess image
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                    img = cv2.resize(img, (48, 48))

                    # Normalize pixel values
                    img = img / 255.0

                    images.append(img)
                    labels.append(emotion_num)
                except Exception as e:
                    print(f"Error loading image {img_path}: {e}")

    # Convert to numpy arrays
    images = np.array(images)
    labels = np.array(labels)

    return images, labels

In [ ]:
# Load training and test data
print("Loading training data...")
train_images, train_labels = load_dataset(train_path)

print("Loading test data...")
test_images, test_labels = load_dataset(test_path)

# Reshape images
train_images = train_images.reshape(-1, 48, 48, 1)
test_images = test_images.reshape(-1, 48, 48, 1)

# IMPORTANT:
# Keep raw labels before one-hot encoding
train_labels_raw = train_labels.copy()
test_labels_raw = test_labels.copy()

# Split BEFORE one-hot encoding
X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    train_images,
    train_labels_raw,
    test_size=0.2,
    random_state=42,
    stratify=train_labels_raw
)

# One-hot encode AFTER split
y_train = to_categorical(y_train_raw, num_classes=7)
y_val = to_categorical(y_val_raw, num_classes=7)

test_labels = to_categorical(test_labels_raw, num_classes=7)

print("\nDataset Statistics:")
print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {test_images.shape}")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_raw),
    y=y_train_raw
)

class_weights = dict(enumerate(class_weights))

print("Class Weights:")
print(class_weights)

In [ ]:
# Create data generator for augmentation
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Fit the generator to your training data
train_datagen.fit(X_train)

In [ ]:
def create_model():
    model = Sequential()

    # First Conv Block
    model.add(Conv2D(64, (3,3), padding='same', activation='relu', input_shape=(48, 48, 1)))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.25))

    # Second Conv Block
    model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.25))

    # Third Conv Block
    model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.25))

    # Fully Connected Layers
    model.add(Flatten())
    model.add(Dense(1024, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(512, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(7, activation='softmax'))  # 7 emotion classes

    return model

# Build and summarize
model = create_model()
model.summary()


In [ ]:
# Define optimizer
optimizer = Adam(learning_rate=0.0001)

# Compile the model
model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Callbacks
checkpoint = ModelCheckpoint(
    'best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [ ]:

# Training parameters
batch_size = 64
epochs = 30

# Train the model
history = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=batch_size),
    steps_per_epoch=len(X_train) // batch_size,
    epochs=epochs,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    callbacks=[checkpoint, early_stopping, reduce_lr],
    verbose=1
)

model.save("emotion_model_final.h5")

print("Training Complete!")
print("Best model saved as: best_model.h5")
print("Final model saved as: emotion_model_final.h5")

In [ ]:
# MODEL EVALUATION

y_pred = model.predict(X_val)

y_pred_classes = np.argmax(y_pred, axis=1)
y_true = y_val_raw

print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred_classes,
        target_names=list(emotion_labels.values())
    )
)

In [ ]:
# CONFUSION MATRIX

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=list(emotion_labels.values()),
    yticklabels=list(emotion_labels.values())
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Emotion Confusion Matrix")

plt.tight_layout()
plt.show()

Epoch 1/20
359/359 [==============================] - 86s 235ms/step - loss: 2.0340 - accuracy: 0.2660 - val_loss: 10.7922 - val_accuracy: 0.1750
Epoch 2/20
359/359 [==============================] - 74s 205ms/step - loss: 1.6356 - accuracy: 0.3417 - val_loss: 1.5938 - val_accuracy: 0.3696
Epoch 3/20
359/359 [==============================] - 74s 206ms/step - loss: 1.5580 - accuracy: 0.3845 - val_loss: 1.4735 - val_accuracy: 0.4286
Epoch 4/20
359/359 [==============================] - 74s 207ms/step - loss: 1.5097 - accuracy: 0.3973 - val_loss: 1.4430 - val_accuracy: 0.4443
Epoch 5/20
359/359 [==============================] - 79s 220ms/step - loss: 1.4639 - accuracy: 0.4179 - val_loss: 1.6122 - val_accuracy: 0.3570
Epoch 6/20
359/359 [==============================] - 79s 220ms/step - loss: 1.3976 - accuracy: 0.4398 - val_loss: 1.4329 - val_accuracy: 0.4455
Epoch 7/20
359/359 [==============================] - 78s 218ms/step - loss: 1.3534 - accuracy: 0.4686 - val_loss: 1.5457 - val_a